# Analyze metric correlations from multiome cohort

In [3]:
import numpy as np
import pandas as pd
import scipy
from scipy.cluster.hierarchy import linkage, leaves_list
import seaborn as sns
import glob
import matplotlib.pyplot as plt
import colorcet as cc

In [4]:
import sys
sys.path.insert(0, '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/scripts/project_utils/')

from analysis_helpers import *

## Create final dataset of metric values

Like the transcriptomics cohort, there are some recalculations.

In [3]:
dirs = ['multiome_gpu', 'multiome_cpu', 'multiome_cpu_neurips']
df = pd.concat([
    pd.read_csv(f'/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/{d}/metrics/all-metrics.csv') for d in dirs
])

ksim = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/ksim_rescue/*.csv')])
df = pd.concat([df, ksim])

align = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/align_multiome/*.csv')])
df = pd.concat([df, align])

corrmse = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/corr_mse_multiome/*.csv')])
df = pd.concat([df, corrmse])

traj = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/traj_multiome/*.csv')])
df = pd.concat([df, traj])

cbe = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/cbe_multiome/*.csv')])
df = pd.concat([df, cbe])

ldfdiff = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/ldfdiff_multiome/*.csv')])
df = pd.concat([df, ldfdiff])

bps = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/bps_multiome/*.csv')])
df = pd.concat([df, bps])

dbich = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/dbich_multiome/*.csv')])
df = pd.concat([df, dbich])

awc = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/awc_multiome/*.csv')])
df = pd.concat([df, awc])

blogreg = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/blogreg_multiome/*.csv')])
df = pd.concat([df, blogreg])

sv = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/sv_multiome/*.csv')])
df = pd.concat([df, sv])

ccc = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/ccc_multiome/*.csv')])
df = pd.concat([df, ccc])

csms = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/csmc_multiome//*.csv')])
df = pd.concat([df, csms])

probe = pd.concat([pd.read_csv(x) for x in glob.glob('/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/recalculate/probe_multiome/*.csv')])
df = pd.concat([df, probe])

mira = pd.read_csv(f'/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/multiome_rescue_mira/metrics/all-metrics.csv')
df = pd.concat([df, mira])

In [4]:
df_rescue = []

with open('../rescue/rescue_dir_multiome_list_cpu.txt', 'r') as f:
    rescue = [s.strip() for s in f.readlines()]
for r in rescue:
    try:
        df_rescue.append(pd.read_csv(glob.glob(r + '/*.csv')[0]))
    except:
        pass
    
with open('../rescue/rescue_dir_multiome_list_gpu.txt', 'r') as f:
    rescue = [s.strip() for s in f.readlines()]
for r in rescue:
    try:
        df_rescue.append(pd.read_csv(glob.glob(r + '/*.csv')[0]))
    except:
        pass
df_rescue = pd.concat(df_rescue)
df = pd.concat([df, df_rescue])

In [5]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df['Metric'] = df['Metric'].str.lower()
df = df.drop_duplicates(
    subset=["Metric", "Dataset", "Integration"], keep='last'
)

In [6]:
metrics = df["Metric"].unique()
datasets = df["Dataset"].unique()
integrations = df["Integration"].unique()

full_index = pd.MultiIndex.from_product(
    [metrics, datasets, integrations],
    names=["Metric", "Dataset", "Integration"]
)
df_full = df.set_index(["Metric", "Dataset", "Integration"]).reindex(full_index).reset_index()

In [7]:
df_full = df_full.sort_values(["Integration", "Dataset", "Metric"]).reset_index(drop=True)

In [8]:
df_full = df_full[df_full.Metric != 'clustacc']

In [9]:
# Some metric values need to be inverted due to earlier bug in code
target_metrics = ['mix', 'kldiv', 'knndiv']
mask = df_full['Metric'].isin(target_metrics)
df_full.loc[mask, 'Value'] = df_full.loc[mask, 'Value'] * -1

In [10]:
df_full = df_full.replace([np.inf, -np.inf], np.nan)

In [11]:
# Rename metrics
df_full['Metric'] = df_full['Metric'].replace({
    'bnentropy': 'bnent',
    'cnentropy': 'cnent',
    'ccentropy': 'ccent',
    'bcentropy': 'bcent',
    'knndiv': 'embdiv',
    'csmc': 'csms',
    'dbi': 'cdbi',
    'ch': 'clustch',
    'cbe': 'cre',
    'bps': 'bp'
})

In [12]:
# Successful computations and total possible computations
(~df_full.Value.isna()).sum(), len(df_full)

(np.int64(10988), 12045)

In [13]:
df_full.to_csv('../results/spreadsheets/metric_multiome_v3.csv')

## Analysis of metrics

In [40]:
df_full = pd.read_csv('../results/spreadsheets/metric_multiome_v3.csv', index_col=0)

In [41]:
df_full2 = df_full[~df_full.Integration.isin([ 'scmomat-1'])]

In [32]:
def rowwise_corr_ignore_nan(X, method='pearson'):
    X = np.asarray(X, float)
    n = X.shape[0]

    corr = np.full((n, n), np.nan)

    for i in range(n):
        for j in range(n):
            mask = ~np.isnan(X[i]) & ~np.isnan(X[j])
            if mask.sum() < 2:
                continue

            xi, xj = X[i, mask], X[j, mask]

            if method == 'pearson':
                corr[i, j] = np.corrcoef(xi, xj)[0, 1]
            elif method == 'spearman':
                corr[i, j] = scipy.stats.spearmanr(xi, xj).statistic
            else:
                raise ValueError("Wrong correlation method specified")
    return corr

In [33]:
bio = ['agree', 'ami', 'ari', 'arif', 'awc', 'bami', 'bari', 'bvm', 'casw', 'ccc', 'cch', 'ccent', 'clustch', 'clisi', 'clustasw', 'clustdbi',
       'cnent', 'comp', 'compact', 'corrmse', 'cdbi', 'f1', 'fknn', 'flogreg', 'fmi', 'homo', 'isoasw', 'isof1',
       'jacc', 'ksim', 'ldfdiff', 'mcc', 'modularity', 'morani', 'nmi', 'nmif', 'nnmap', 'ocs', 'purity', 'rknn',
       'scgraph', 'separation', 'traj', 'vi', 'vm',]
batch = ['adgbd', 'align', 'basw', 'batchari', 'bcent', 'bknn', 'blogreg', 'bnent', 'bp', 'bras', 'cre', 'cilisi', 'csms', 'gc', 'ilisi',  'kbet', 'kbetall',
         'kldiv', 'embdiv', 'mix', 'pcr', 'pptpc', 'probe', 'proberf', 'probelr', 'pstdb', 'sv']
both = ['pptpc', 'pstdb', 'blogreg', 'bknn', 'gc']
all_metrics = sorted(bio + batch)
len(bio),len(batch)

(45, 27)

In [34]:
bio_df = df_full[df_full['Metric'].isin(bio + both)]
batch_df = df_full[df_full['Metric'].isin(batch + both)]

In [35]:
bio_corrs = rowwise_corr_ignore_nan(np.array(bio_df.groupby('Metric')['Value'].apply(list).to_list()), method='spearman')
batch_corrs = rowwise_corr_ignore_nan(np.array(batch_df.groupby('Metric')['Value'].apply(list).to_list()), method='spearman')
all_corrs = rowwise_corr_ignore_nan(np.array(df_full.groupby('Metric')['Value'].apply(list).to_list()), method='spearman')

## Metric correlations w/o averaging

These plots are not used in the manuscript.

In [ ]:
dist = 1 - bio_corrs

Z = linkage(dist, method='average')
order = leaves_list(Z)

bio_corrs_sorted = bio_corrs[order][:, order]
bio_sorted = [(bio+both)[i] for i in order]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(bio_corrs_sorted, aspect='auto', cmap='PiYG', vmax=1, vmin=-1)
plt.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(np.arange(len(bio_sorted)))
ax.set_yticks(np.arange(len(bio_sorted)))
ax.set_xticklabels(bio_sorted, rotation=90)
ax.set_yticklabels(bio_sorted)

ax.set_title("Bioconservation metrics")
plt.tight_layout()
plt.show()

In [ ]:
dist = 1 - batch_corrs

Z = linkage(dist, method='average')
order = leaves_list(Z)

batch_corrs_sorted = batch_corrs[order][:, order]
batch_sorted = [batch[i] for i in order]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(batch_corrs_sorted, aspect='auto', cmap='PiYG', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(np.arange(len(batch_sorted)))
ax.set_yticks(np.arange(len(batch_sorted)))
ax.set_xticklabels(batch_sorted, rotation=90)
ax.set_yticklabels(batch_sorted)

ax.set_title("Batch correction metrics")
plt.tight_layout()
plt.show()

In [ ]:
dist = 1 - all_corrs

Z = linkage(dist, method='average')
order = leaves_list(Z)

all_corrs_sorted = all_corrs[order][:, order]
all_metrics_sorted = [all_metrics[i] for i in order]

fig, ax = plt.subplots(figsize=(11, 10))

im = ax.imshow(all_corrs_sorted, aspect='auto', cmap='PiYG', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(np.arange(len(all_metrics_sorted)))
ax.set_yticks(np.arange(len(all_metrics_sorted)))
ax.set_xticklabels(all_metrics_sorted, rotation=90)
ax.set_yticklabels(all_metrics_sorted)

for tick in ax.get_xticklabels():
    if tick.get_text() in bio:
        tick.set_color('blue')
    else:
        tick.set_color('red')

for tick in ax.get_yticklabels():
    if tick.get_text() in bio:
        tick.set_color('blue')
    else:
        tick.set_color('red')

ax.set_title("All metrics correlation")
plt.tight_layout()
plt.show()

## Metric correlations average per dataset

The plots generated in Python are not used in the manuscript. The correlations are written out to be plotted using ComplexHeatmap in R.

In [ ]:
df_pivoted = bio_df.pivot(index=['Dataset', 'Integration'], columns='Metric', values='Value')
correlations = df_pivoted.groupby('Dataset').corr(method='spearman')

avg_corr = correlations.groupby(level=1).median()
# avg_corr.to_csv('../results/spreadsheets/multiome_bioconservation_corr.csv')

dist = 1 - avg_corr.to_numpy()

Z = linkage(dist, method='average')

order = leaves_list(Z)

avg_corrs_sorted = avg_corr.to_numpy()[order][:, order]
avg_sorted = [avg_corr.index[i] for i in order]

fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(avg_corrs_sorted, aspect='auto', cmap='PiYG', vmax=1, vmin=-1)
plt.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(np.arange(len(avg_sorted)))
ax.set_yticks(np.arange(len(avg_sorted)))
ax.set_xticklabels(avg_sorted, rotation=90)
ax.set_yticklabels(avg_sorted)

ax.set_title("Bioconservation metrics")
plt.tight_layout()
plt.show()

In [ ]:
g = sns.clustermap(avg_corr, method='average', figsize=(18, 18), 
                   metric='euclidean', cmap='PiYG', vmin=-1, vmax=1, 
                   dendrogram_ratio=0.1)
g.ax_row_dendrogram.set_visible(False)

x_labels = g.ax_heatmap.get_xticklabels()
for label in x_labels:
    text = label.get_text()
    label.set_text(text.upper())

y_labels = g.ax_heatmap.get_yticklabels()
for label in y_labels:
    text = label.get_text()
    label.set_text(text.upper())


g.ax_heatmap.set_xticklabels(x_labels)
g.ax_heatmap.set_yticklabels(y_labels)

plt.show()

In [ ]:
df_pivoted = batch_df.pivot(index=['Dataset', 'Integration'], columns='Metric', values='Value')
correlations = df_pivoted.groupby('Dataset').corr(method='spearman')

avg_corr = correlations.groupby(level=1).median()
# avg_corr.to_csv('../results/spreadsheets/multiome_batch_corr.csv')

dist = 1 - avg_corr.to_numpy()

Z = linkage(dist, method='average')

order = leaves_list(Z)

avg_corrs_sorted = avg_corr.to_numpy()[order][:, order]
avg_sorted = [avg_corr.index[i] for i in order]

fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(avg_corrs_sorted, aspect='auto', cmap='PiYG', vmax=1, vmin=-1)
plt.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(np.arange(len(avg_sorted)))
ax.set_yticks(np.arange(len(avg_sorted)))
ax.set_xticklabels(avg_sorted, rotation=90)
ax.set_yticklabels(avg_sorted)

ax.set_title("Batch correction metrics")
plt.tight_layout()
plt.show()

In [ ]:
df_pivoted = df_full.pivot(index=['Dataset', 'Integration'], columns='Metric', values='Value')
correlations = df_pivoted.groupby('Dataset').corr(method='spearman')
avg_corr = correlations.groupby(level=1).median()
avg_corr.to_csv('../results/spreadsheets/multiome_overall_corr_altered.csv')

dist = 1 - avg_corr.to_numpy()

Z = linkage(dist, method='average', optimal_ordering=False)

order = leaves_list(Z)

avg_corrs_sorted = avg_corr.to_numpy()[order][:, order]
avg_sorted = [avg_corr.index[i] for i in order]

fig, ax = plt.subplots(figsize=(10, 9))

im = ax.imshow(avg_corrs_sorted, aspect='auto', cmap='PiYG', vmax=1, vmin=-1)
plt.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(np.arange(len(avg_sorted)))
ax.set_yticks(np.arange(len(avg_sorted)))
ax.set_xticklabels(avg_sorted, rotation=90)
ax.set_yticklabels(avg_sorted)
for tick in ax.get_xticklabels():
    if tick.get_text() in both:
        tick.set_color('orange')
    elif tick.get_text() in bio:
        tick.set_color('blue')
    else:
        tick.set_color('red')

for tick in ax.get_yticklabels():
    if tick.get_text() in both:
        tick.set_color('orange')
    elif tick.get_text() in bio:
        tick.set_color('blue')
    else:
        tick.set_color('red')
ax.set_title("")
plt.tight_layout()
plt.show()

In [ ]:
g = sns.clustermap(avg_corr, method='average', figsize=(20, 20), metric='euclidean', cmap='PiYG', vmin=-1, vmax=1)
reordered_labels = g.ax_heatmap.get_xticklabels()

for label in reordered_labels:
    text = label.get_text()
    if text in both:
        label.set_color('orange')
    elif text in bio:
        label.set_color('blue')
    elif text in batch:
        label.set_color('red')

for label in g.ax_heatmap.get_yticklabels():
    text = label.get_text()
    if text in both:
        label.set_color('orange')
    elif text in bio:
        label.set_color('blue')
    elif text in batch:
        label.set_color('red')

plt.show()

In [48]:
df_pivoted = df_full.pivot(index=['Dataset', 'Integration'], columns='Metric', values='Value')
correlations = df_pivoted.groupby('Dataset').corr(method='spearman')
avg_corr1 = correlations.groupby(level=1).median()

df_pivoted = df_full2.pivot(index=['Dataset', 'Integration'], columns='Metric', values='Value')
correlations = df_pivoted.groupby('Dataset').corr(method='spearman')
avg_corr2 = correlations.groupby(level=1).median()

In [55]:
np.median(avg_corr1.loc[bio, batch].to_numpy())

np.float64(-0.19642857142857142)

In [56]:
np.median(avg_corr2.loc[bio, batch].to_numpy())

np.float64(-0.0945054945054945)